# Nemotron Reasoning Challenge - High Quality LoRA Training

Uses programmatic solvers to generate real CoT traces, then trains LoRA adapter on full dataset.

In [ ]:
# OFFLINE INSTALL
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index",
    "--find-links", "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages",
    "datasets", "trl", "--ignore-installed"
], stderr=subprocess.DEVNULL)

import datasets, trl
print("datasets:", datasets.__version__, "  trl:", trl.__version__)

In [ ]:
# IMPORTS
import os, sys, stat, shutil, gc, zipfile, glob
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
except Exception:
    pass

In [ ]:
# TRITON FIX
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    print("Triton ptxas fix applied.")

In [ ]:
# HYPERPARAMETERS - OPTIMIZED FOR QUALITY
LORA_RANK      = 32
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.05
MAX_SEQ_LEN    = 1536
NUM_EPOCHS     = 3
GRAD_ACCUM     = 8
LR             = 5e-5
WARMUP_RATIO   = 0.05
OUTPUT_DIR     = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config: rank={LORA_RANK}, alpha={LORA_ALPHA}, epochs={NUM_EPOCHS}, lr={LR}")

In [ ]:
# LOAD DATA
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

candidate_train_paths = [
    os.environ.get("TRAIN_CSV"),
    "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv",
    "/kaggle/input/nvidia-nemotron-3-nano-30b-reasoning-challenge/train.csv",
    "/kaggle/input/train.csv",
    "/kaggle/working/train.csv",
    os.path.join(os.getcwd(), "train.csv"),
]
candidate_train_paths.extend(sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)))
train_path = next((p for p in candidate_train_paths if p and os.path.exists(p)), None)
if train_path is None:
    raise FileNotFoundError(f"train.csv not found")

train_df = pl.read_csv(train_path)
print(f"Total samples: {len(train_df)}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, trust_remote_code=True, model_max_length=MAX_SEQ_LEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# GENERATE HIGH-QUALITY CoT TRAINING DATA USING SOLVERS
sys.path.insert(0, '/kaggle/input/nemotron-v2')
try:
    from solvers import build_training_text
    print("Using programmatic solvers for real CoT traces!")

    def process_example(row):
        text = build_training_text(row["prompt"], row["answer"])
        return {"text": text}

    hf_dataset = Dataset.from_pandas(train_df.to_pandas())
    hf_dataset = hf_dataset.map(process_example, remove_columns=hf_dataset.column_names)
    print(f"Generated {len(hf_dataset)} high-quality training samples with real CoT")
except Exception as e:
    print(f"Solvers failed: {e}, using fallback")

    SYSTEM_PROMPTS = {
        "bit": "You are an expert in binary operations. Discover the hidden bit transformation rule from examples. Reason inside <think> tags. Final answer in \\boxed{}.",
        "cipher": "You are an expert cryptographer. Build a substitution table and decrypt. Reason inside <think> tags. Final answer in \\boxed{}.",
        "roman": "You are an expert in Roman numerals. M=1000 CM=900 D=500 CD=400 C=100 XC=90 L=50 XL=40 X=10 IX=9 V=5 IV=4 I=1. Convert inside <think>. Final answer in \\boxed{}.",
        "gravity": "You are a physicist. Use d = 0.5*g*t^2. Find g from examples, then solve for target. Show work in <think>. Final answer in \\boxed{}.",
        "unit": "You are an expert in unit conversions. Find conversion factor from examples, verify, then apply. Show work in <think>. Final answer in \\boxed{}.",
        "symbol": "You are an expert in symbolic patterns. Identify transformation rules and apply. Reason in <think>. Final answer in \\boxed{}.",
        "other": "You are an expert reasoning assistant. Analyze patterns, reason in <think>, give final answer in \\boxed{}.",
    }
    COT_TRACES = {
        "bit": "Analyzing XOR masks, rotations, and bit permutations across all examples. The rule is consistent. Applying to target.",
        "cipher": "Building substitution table from examples. Verifying consistency. Decrypting test text.",
        "roman": "Converting using standard Roman numeral values, largest to smallest.",
        "gravity": "Computing g = 2d/t^2 from examples. Verifying consistency. Computing d = 0.5*g*t^2.",
        "unit": "Computing conversion factor = output/input. Verifying. Applying to target.",
        "symbol": "Analyzing character mappings and positional patterns. Applying rule.",
        "other": "Analyzing pattern from examples. Applying to target.",
    }
    def detect_family(p):
        t = p.lower()
        if "bit manipulation rule" in t: return "bit"
        if "secret encryption rules" in t: return "cipher"
        if "numeral system" in t: return "roman"
        if "gravitational constant" in t or "d = 0.5*g*t^2" in t: return "gravity"
        if "unit conversion" in t: return "unit"
        if "transformation rules is applied" in t: return "symbol"
        return "other"
    def build_text(row):
        prompt, answer = row["prompt"], str(row["answer"]).strip()
        family = detect_family(prompt)
        system = SYSTEM_PROMPTS[family]
        cot = COT_TRACES[family]
        text = (
            f"<|system|>\n{system}\n<|endoftext|>\n"
            f"<|user|>\n{prompt}\nPut your final answer inside \\boxed{{}}.\n<|endoftext|>\n"
            f"<|assistant|>\n"
            f"<think>\n{cot}\n</think>\n\n"
            f"\\boxed{{{answer}}}\n<|endoftext|>"
        )
        return {"text": text}

    hf_dataset = Dataset.from_pandas(train_df.to_pandas())
    hf_dataset = hf_dataset.map(build_text, remove_columns=hf_dataset.column_names)
    print(f"Generated {len(hf_dataset)} training samples (fallback CoT)")

# sanity check
sample = hf_dataset[0]["text"]
assert "<think>" in sample and "</think>" in sample, "Think tags missing!"
assert "\\boxed{" in sample, "Boxed answer missing!"
print(f"Format check passed.\\n\\nSample preview:\\n{sample[:400]}\\n...")

In [ ]:
# LOAD MODEL
import unittest.mock as mock

for _mod in [
    'cutlass', 'cutlass.cute',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
    'mamba_ssm.ops.triton.mamba3',
    'mamba_ssm.ops.triton.mamba3.mamba3_mimo_rotary_step',
    'mamba_ssm.modules.mamba3',
]:
    sys.modules[_mod] = mock.MagicMock()

import importlib
for _mod_name in list(sys.modules.keys()):
    if 'mamba_ssm' in _mod_name and 'mamba3' not in _mod_name and 'cute' not in _mod_name:
        try:
            importlib.reload(sys.modules[_mod_name])
        except Exception:
            pass

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.config.use_cache = False
print(f"Model loaded. Vocab size: {len(tokenizer)}")

gc.collect()
torch.cuda.empty_cache()

for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}")

In [ ]:
# LORA SETUP
lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = "all-linear",
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# TRAINING
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    tokenizer=tokenizer,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        max_seq_length=MAX_SEQ_LEN,
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        optim="adamw_torch",
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        packing=False,
    ),
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
print("Training complete!")

In [ ]:
# CREATE SUBMISSION.ZIP
save_path = "/kaggle/working/adapter"
os.makedirs(save_path, exist_ok=True)

# Copy adapter files
for fname in os.listdir(OUTPUT_DIR):
    if fname.startswith("adapter"):
        shutil.copy2(os.path.join(OUTPUT_DIR, fname), os.path.join(save_path, fname))

# Also copy README.md if needed
if not os.path.exists(os.path.join(save_path, "README.md")):
    with open(os.path.join(save_path, "README.md"), "w") as f:
        f.write("# Nemotron LoRA Adapter\n")

# Create submission zip
zip_path = "/kaggle/working/submission.zip"
os.chdir(save_path)
subprocess.run(f"zip -r {zip_path} adapter_config.json adapter_model.safetensors README.md",
               shell=True, check=True)
print("submission.zip created!")